In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
%cd drive/MyDrive/

In [ ]:
!rm -rf FPL_forecast
!git clone https://github.com/bragehs/FPL_forecast.git

In [ ]:
%cd FPL_forecast/predictor/

In [3]:
file_path = '/content/drive/MyDrive/colab_fpl'
file_path

'/content/drive/MyDrive/colab_fpl'

In [4]:
import os
import torch
from training import train_model, hyperparameter_tuning
from model import HybridLSTMAttn

In [5]:
file_path = os.getcwd() + "/processed_data"
file_path

'/Users/bragehs/Documents/FPL_forecast/predictor/processed_data'

In [6]:
X_train = torch.load(file_path + "/X_train.pt", weights_only=True)
y_train = torch.load(file_path + "/y_train.pt", weights_only=True)
X_val = torch.load(file_path + "/X_val.pt", weights_only=True)
y_val = torch.load(file_path + "/y_val.pt", weights_only=True)

print(f"Train sequences: {X_train.shape}, Targets: {y_train.shape}")

Train sequences: torch.Size([72339, 5, 37]), Targets: torch.Size([72339, 1])


In [ ]:
# Hyperparameter tuning
best_params = hyperparameter_tuning(X_train, y_train, X_val, y_val, epochs=20, n_trials=20, transform=False, num_workers=4)

# Full training with best hyperparameters
print("\nTraining final model with best hyperparameters...")
adv_model = HybridLSTMAttn(input_dim=X_train.shape[-1], hidden_dim=best_params['hidden_dim'],
                            output_dim=1, num_layers=best_params['num_layers'],
                            dropout=best_params['dropout'], transformer_layers=best_params['transformer_layers'])


train_model(
    adv_model,
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    learning_rate=best_params['learning_rate'],
    weight_decay=best_params['weight_decay'],
    batch_size=best_params['batch_size'],
    epochs=100,  # Full training
    verbose=2,
    transform=False,
    num_workers=4,
)

Running random search with 20 trials...

Trial 1/20
Params: {'learning_rate': 0.0005, 'hidden_dim': 256, 'weight_decay': 1e-06, 'num_layers': 3, 'dropout': 0.3, 'transformer_layers': 2, 'batch_size': 64}


libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x11a72d620>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1603, in __del__
    def __del__(self):

  File "/opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/torch/utils/data/_utils/signal_handling.py", line 73, in handler
    _error_if_any_worker_fails()
RuntimeError: DataLoader worker (pid 47079) is killed by signal: Abort trap: 6. 


KeyboardInterrupt: 